### Prepare Data

In [4]:
#%pip install pandas openpyxl

In [10]:
import pandas as pd

url = "https://raw.githubusercontent.com/aschan12/DADS5250Hackathon/main/DADS5250_Hackathon_Google_Data.xlsx"

# 2. Read directly into a DataFrame
df_salary_threshold = pd.read_excel(url, sheet_name="Agent Salary and Threshold", engine="openpyxl")
df_adv_sign_ups = pd.read_excel(url, sheet_name="Advertiser Sign-ups", engine="openpyxl")
df_exist_staff = pd.read_excel(url, sheet_name="Existing Agent Staffing", engine="openpyxl")

print(df_salary_threshold.head())
print(df_adv_sign_ups.head())
print(df_exist_staff.head())

   Country  Annual_Agent_Salary_USD  Advertiser_Eligibility_Threshold_USD
0      USA                    77721                                 35000
1       UK                    51650                                 27000
2  Germany                    51328                                 26000
3    India                    14118                                 13000
4   Brazil                    27950                                 29000
  Advertiser_ID Country         Sign_Up_Date  Projected_Annual_Budget
0       A000001     USA  2024-01-01 00:00:00                    84376
1       A000002     USA  2024-01-01 00:00:00                    57297
2       A000003     USA  2024-01-01 00:00:00                    34081
3       A000004     USA  2024-01-01 00:00:00                   154506
4       A000005     USA  2024-01-01 00:00:00                    23604
   Country  Existing_Agent_Count
0      USA                   652
1       UK                   589
2  Germany                   602
3   

<b> Eligible Ads:  
If Annual Budget > Threshold, then Eligible.  
Remove all Feb 29, 2025 </b>

In [23]:
# Left join the first two tables
eligible_ad = pd.merge(df_adv_sign_ups, df_salary_threshold, on='Country', how='left')
eligible_ad = eligible_ad.drop(columns=['Annual_Agent_Salary_USD'])

# Find eligible advertisers
eligible_ad['Eligible'] = (eligible_ad['Projected_Annual_Budget']> eligible_ad['Advertiser_Eligibility_Threshold_USD'])
eligible_ad = eligible_ad[eligible_ad['Eligible'] == True]

# Drop all on Feb 29, 2025 -> These are the ones that create Error when converting to_datetime
eligible_ad['Sign_Up_Date'] = pd.to_datetime(eligible_ad['Sign_Up_Date'], errors='coerce')
# 2. Drop all rows where 'Sign_Up_Date' became NaT
eligible_ad = eligible_ad.dropna(subset=['Sign_Up_Date'])

eligible_ad.head()

,Advertiser_ID,Country,Sign_Up_Date,Projected_Annual_Budget,Advertiser_Eligibility_Threshold_USD,Eligible
0,A000001,USA,2024-01-01,84376,35000,True
1,A000002,USA,2024-01-01,57297,35000,True
3,A000004,USA,2024-01-01,154506,35000,True
7,A000008,USA,2024-01-01,104967,35000,True
8,A000009,USA,2024-01-01,44165,35000,True


In [179]:
# Past Daily Demands
# daily_demand = eligible_ad.groupby(['Country', 'Sign_Up_Date']).size().reset_index(name='eligible_count')

#daily_demand.head()

In [205]:
# Monthly totals
#eligible_ad['Month'] = (eligible_ad['Sign_Up_Date'].dt.to_period('M'))
#monthly_demand = (eligible_ad.groupby(['Country', 'Month']).size())

#monthly_demand.head()

## Logic  
### When an eligible advertiser comes in, they sit in a waiting pool.  
### Path 1: If there's an agent available, they are assigned to it. Supported for 60 days, and 'graduate'.  
### Path 2: No available agent, they will continue waiting. Everyday, if there's an agent available, they will be assigned. After 60 days if they are still unassigned, they are removed from the pool.

## Components:
### 1. Simulator by country - then a loop to loop through all countries
### 2. Staffing schedule
### 3. Evaluate optimization function

In [204]:
def simulate_country(advertisers, country, staffing_plan, start_date="2024-01-01", end_date="2024-12-31"):
#def simulate_country(advertisers, country, agent_count, start_date="2024-01-01", end_date="2024-12-31"):
    df = advertisers[advertisers['Country'] == country].copy()

    #df["signup_date"] = pd.to_datetime(df["signup_date"])

    df['status'] = 'NOT_ARRIVED'
    df['assignment_date'] = pd.NaT
    df['graduation_date'] = pd.NaT
    df['expiration_date'] = pd.NaT

    daily_results = []

    dates = pd.date_range(start=start_date, end=end_date, freq="D")

    for current_date in dates:

        # Enhancement 1a: dynamic staffing
        current_month = current_date.to_period("M")
        staffing_row = staffing_plan[(staffing_plan["Country"] == country) & (staffing_plan["Month"] == current_month)]
        if staffing_row.empty:
            raise ValueError(f"No staffing data found for " 
                             f"{country}, {current_month}")
        agent_count = int(staffing_row["Agents"].iloc[0])

        # 1. Graduate advertisers
        graduating = ((df["status"] == "ASSIGNED")  & (df["graduation_date"] <= current_date))
        graduated_today = graduating.sum()
        df.loc[graduating, "status"] = "GRADUATED"

        # 2. Add today's advertisers
        arrivals = ((df["Sign_Up_Date"] == current_date) & (df["status"] == "NOT_ARRIVED") )
        new_eligible_today = arrivals.sum()
        df.loc[arrivals, "status"] = "WAITING"

        # 3. Expire old waiting advertisers
        expired = ((df["status"] == "WAITING") & (current_date > df["Sign_Up_Date"]+ pd.Timedelta(days=60)))
        expired_today = expired.sum()
        df.loc[expired, "status"] = "EXPIRED"
        df.loc[expired, "expiration_date"] = current_date

        # 4. Calculate capacity
        total_capacity = agent_count * 10
        currently_supported = (df["status"] == "ASSIGNED").sum()
        available_capacity = max(0, total_capacity - currently_supported)

        # Enhancement 1b: raise error if staff reduction causes currently supported advertisers to suddenly go without an agent
        if currently_supported > total_capacity:
            raise ValueError(f"Infeasible staffing plan: "
                             f"{Country} on {current_date.date()} " f"has {currently_supported} supported advertisers " 
                            f"but only {total_capacity} available capacity.")

        # 5. First-In First-Out assignment
        waiting = (df[df["status"] == "WAITING"].sort_values(["Sign_Up_Date", "Advertiser_ID"]))
        to_assign = waiting.head(available_capacity)
        assigned_today = len(to_assign)
        if assigned_today > 0:
            df.loc[to_assign.index,"status"] = "ASSIGNED"
            df.loc[to_assign.index, "assignment_date"] = current_date
            df.loc[to_assign.index, "graduation_date"] = (current_date + pd.Timedelta(days=60)            )

        # 6. Daily results
        supported = (df["status"] == "ASSIGNED").sum()
        waiting_count = (df["status"] == "WAITING").sum()
        utilization = (supported / total_capacity if total_capacity > 0 else 0)
        daily_results.append({
            "date": current_date,
            "Country": country,
            "agents": agent_count,
            "capacity": total_capacity,
            "new_eligible": new_eligible_today,
            "assigned_today": assigned_today,
            "graduated_today": graduated_today,
            "expired_today": expired_today,
            "supported": supported,
            "waiting": waiting_count,
            "utilization": utilization
        })

    return (df, pd.DataFrame(daily_results))

In [182]:
#staffing_plan = pd.DataFrame({
#    "country": ["USA"] * 12,
#    "month": pd.period_range(
#        "2024-01",
#        "2024-12",
#        freq="M"
#    ),
#    "agents": [
#        50, 50, 55, 55, 60, 60, 60, 60, 58, 58, 56, 60]
#})

#print(staffing_plan)

In [101]:
#us_agents = df_exist_staff.loc[df_exist_staff["Country"] == "USA","Existing_Agent_Count"].iloc[0]

#us_advertisers, us_daily = simulate_country(eligible_ad, country="USA", agent_count=us_agents)
#us_daily.head()

In [173]:
def build_staffing_schedule(initial_agents, decisions):

    decisions = decisions.sort_values("Month").copy()
    active_agents = initial_agents

    previous_hires = 0
    previous_terminations = 0

    results = []

    for _, row in decisions.iterrows():

        # Decisions from previous month
        # take effect this month
        active_agents += previous_hires
        active_agents -= previous_terminations

        if active_agents < 0:
            raise ValueError(f"Active agents became negative " f"in {row['Month']}")

        results.append({
            "Country": row["Country"],
            "Month": row["Month"],
            "Agents": active_agents,
            "Agents_Ramping": int(row["Hires"]),
            "Hires": int(row["Hires"]),
            "Terminations_Notices": int(
                row["Terminations"]
            )
        })

        previous_hires = int(row["Hires"])
        previous_terminations = int(row["Terminations"])

    staffing_plan = pd.DataFrame(results)
    staffing_plan = staffing_plan.merge(df_salary_threshold[["Country", "Annual_Agent_Salary_USD"]],
                                        on="Country", how="left")
    staffing_plan['Monthly_Salary'] = staffing_plan['Annual_Agent_Salary_USD']/12
    staffing_plan["Paid_Agents"] = (staffing_plan["Agents"] + staffing_plan["Agents_Ramping"])
    staffing_plan["Salary_Cost"] = (staffing_plan["Paid_Agents"] * staffing_plan["Monthly_Salary"])
    staffing_plan["Termination_Cost"] = (staffing_plan["Terminations_Notices"] * staffing_plan["Annual_Agent_Salary_USD"]* 0.40)
    staffing_plan["Total_Cost"] = (staffing_plan["Salary_Cost"] + staffing_plan["Termination_Cost"])

    return staffing_plan

In [202]:
def evaluate_plan(advertiser_results, daily_results, staffing_plan):

    completed = advertiser_results[advertiser_results["assignment_date"].notna()].copy()
    completed["wait_days"] = (completed["assignment_date"] - completed["Sign_Up_Date"]).dt.days
    max_wait = completed["wait_days"].max()
    avg_wait = completed["wait_days"].mean()
    p90_wait = completed["wait_days"].quantile(0.90)
    expired = (advertiser_results["status"] == "EXPIRED").sum()
    expiration_rate = (expired / len(advertiser_results))
    avg_utilization = (daily_results["utilization"].mean())
    peak_utilization = (daily_results["utilization"].max())

    return {
        "advertisers_supported": len(completed),
        "advertisers_expired": int(expired),
        "average_wait_days": avg_wait,
        "p90_wait_days": p90_wait,
        "average_utilization": avg_utilization,
        "expired_advertisers": expired_count,
        "expiration_rate": expiration_rate,
        "total_staffing_cost": staffing_plan["Total_Cost"].sum()
    }
    

### Find Baseline For All Countries 
<b> Baseline = Agent count stays the same </b>

In [197]:
all_staffing_plans = []
all_advertiser_results = []
all_daily_results = []
all_baseline_summary = []

for _, row in df_exist_staff.iterrows():

    country = row["Country"]
    agents = int(row["Existing_Agent_Count"])
    baseline_decisions = pd.DataFrame({
        "Country": [country] * 12,
        "Month": pd.period_range(
            "2024-01",
            "2024-12",
            freq="M"),
        "Hires": [0] * 12,
        "Terminations": [0] * 12
    })

    baseline_staffing_plan = build_staffing_schedule(initial_agents=agents, decisions=baseline_decisions)
    advertiser_result, daily_result = simulate_country(eligible_ad, country=country, staffing_plan=baseline_staffing_plan, start_date="2024-01-01", end_date="2024-12-31")

    all_advertiser_results.append(advertiser_result)
    all_daily_results.append(daily_result)

    evaluate_plan(advertiser_results= advertiser_result,daily_results = daily_result, staffing_plan = baseline_staffing_plan)
    

simulation_daily = pd.concat(all_daily_results, ignore_index=True)
simulation_advertisers = pd.concat(all_advertiser_results,ignore_index=True) 


In [201]:
simulation_advertisers.head()

,Advertiser_ID,Country,Sign_Up_Date,Projected_Annual_Budget,Advertiser_Eligibility_Threshold_USD,Eligible,Month,status,assignment_date,graduation_date,expiration_date
0,A000001,USA,2024-01-01,84376,35000,True,2024-01,GRADUATED,2024-01-01,2024-03-01,NaT
1,A000002,USA,2024-01-01,57297,35000,True,2024-01,GRADUATED,2024-01-01,2024-03-01,NaT
2,A000004,USA,2024-01-01,154506,35000,True,2024-01,GRADUATED,2024-01-01,2024-03-01,NaT
3,A000008,USA,2024-01-01,104967,35000,True,2024-01,GRADUATED,2024-01-01,2024-03-01,NaT
4,A000009,USA,2024-01-01,44165,35000,True,2024-01,GRADUATED,2024-01-01,2024-03-01,NaT


### Now that we have the three functions, we can start building for 2026.
#### The goal is to predict staffing for each country for 2026.
#### First, forecast 2026 demand based on 2024-2025 data.
#### Next, make two Staffing schedules: one using no change (baseline), second using GenAI
#### Put both through simulator and evaluate
#### Compare both performance

In [206]:
import numpy as np
import pandas as pd

# Monthly totals
eligible_ad['Month'] = (eligible_ad['Sign_Up_Date'].dt.to_period('M'))
monthly_demand = (eligible_ad.groupby(['Country', 'Month']).size())

monthly_demand.head()

Country    Month  
Argentina  2024-01    53
           2024-02    66
           2024-03    69
           2024-04    71
           2024-05    67
dtype: int64

In [ ]:
def forecast_2026_monthly_demand(monthly_history, minimum_growth=-0.20, maximum_growth=0.30):
    pivot = monthly_history.pivot_table(
        index=[
            "Country",
            "Month_Number"
        ],
        columns="Year",
        values="Eligible_Advertisers",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    # Ensure both columns exist
    if 2024 not in pivot.columns:
        pivot[2024] = 0

    if 2025 not in pivot.columns:
        pivot[2025] = 0

    pivot = pivot.rename(
        columns={
            2024: "Eligible_2024",
            2025: "Eligible_2025"
        }
    )

    # Calculate historical growth
    pivot["Historical_Growth"] = np.where(
        pivot["Eligible_2024"] > 0,
        (
            pivot["Eligible_2025"]
            - pivot["Eligible_2024"]
        )
        / pivot["Eligible_2024"],
        0
    )

    # Prevent a small historical value from producing
    # an unrealistic 2026 forecast
    pivot["Applied_Growth"] = (
        pivot["Historical_Growth"]
        .clip(
            lower=minimum_growth,
            upper=maximum_growth
        )
    )

    pivot["Forecast_Eligible_2026"] = (
        pivot["Eligible_2025"]
        * (
            1
            + pivot["Applied_Growth"]
        )
    ).round().astype(int)

    pivot["Month"] = pd.PeriodIndex(
        [
            f"2026-{month:02d}"
            for month in pivot["Month_Number"]
        ],
        freq="M"
    )

    return pivot[
        [
            "Country",
            "Month",
            "Month_Number",
            "Eligible_2024",
            "Eligible_2025",
            "Historical_Growth",
            "Applied_Growth",
            "Forecast_Eligible_2026"
        ]
    ].sort_values(
        ["Country", "Month"]
    ).reset_index(drop=True)

In [ ]:
from pydantic import BaseModel, Field
from typing import List
import pandas as pd


class MonthlyStaffingDecision(BaseModel):
    Month: str = Field(
        description="Month in YYYY-MM format"
    )

    Hires: int = Field(
        ge=0,
        description="Number of agents hired during this month"
    )

    Terminations: int = Field(
        ge=0,
        description="Number of termination notices issued this month"
    )

    Reason: str = Field(
        description="Short explanation for the staffing decision"
    )


class StaffingRecommendation(BaseModel):
    Country: str

    Decisions: List[MonthlyStaffingDecision]

    Overall_Strategy: str = Field(
        description="Brief overall explanation of the staffing strategy"
    )

In [187]:
baseline_decisions = pd.DataFrame({
    "Country": ["USA"] * 12,
    "Month": pd.period_range(
        "2024-01",
        "2024-12",
        freq="M"
    ),
    "Hires": [0] * 12,
    "Terminations": [0] * 12
})
#baseline_decisions.head()

# Get staffing plan
baseline_staffing_plan = build_staffing_schedule(initial_agents=50, decisions=baseline_decisions)
us_baseline_ad, us_baseline_daily =  simulate_country(advertisers=eligible_ad, country="USA", staffing_plan=baseline_staffing_plan,start_date="2024-01-01", end_date="2024-12-31")


#baseline_staffing_plan.head()

In [209]:
us_baseline_daily.head()

,date,country,agents,capacity,new_eligible,assigned_today,graduated_today,expired_today,supported,waiting,utilization
0,2024-01-01,USA,50,500,53,53,0,0,53,0,0.106
1,2024-01-02,USA,50,500,30,30,0,0,83,0,0.166
2,2024-01-03,USA,50,500,57,57,0,0,140,0,0.280
3,2024-01-04,USA,50,500,66,66,0,0,206,0,0.412
4,2024-01-05,USA,50,500,29,29,0,0,235,0,0.470


In [203]:
evaluate_plan(advertiser_results= us_baseline_ad,daily_results = us_baseline_daily, staffing_plan = baseline_staffing_plan )

{'advertisers_supported': 3262,
 'advertisers_expired': 6940,
 'average_wait_days': 46.18761496014715,
 'p90_wait_days': 60.0,
 'average_utilization': 0.9869125683060108,
 'expired_advertisers': 6940,
 'expiration_rate': 0.26626764886433396,
 'total_staffing_cost': 20120535.0}

In [194]:
baseline_summary

{'country': 'USA',
 'eligible_advertisers': 26064,
 'supported_advertisers': 3262,
 'expired_advertisers': 6940,
 'expiration_rate': 0.26626764886433396,
 'average_wait_days': 46.18761496014715,
 'p90_wait_days': 60.0,
 'average_utilization': 0.9869125683060108,
 'total_staffing_cost': 3886050.0}

In [193]:
#baseline_staffing_plan.head()
#us_baseline_daily.tail()

assigned = us_baseline_ad[us_baseline_ad["assignment_date"].notna()].copy()
assigned["wait_days"] = (assigned["assignment_date"] - assigned["Sign_Up_Date"]).dt.days
avg_wait = assigned["wait_days"].mean()

p90_wait = assigned["wait_days"].quantile(0.90)

max_wait = assigned["wait_days"].max()
expired_count = (us_baseline_ad["status"] == "EXPIRED").sum()
expiration_rate = (expired_count / len(us_baseline_ad))
average_utilization = (us_baseline_daily["utilization"].mean())
peak_utilization = (us_baseline_daily["utilization"].max())

baseline_summary = {
    "country": "USA",

    "eligible_advertisers":
        len(us_baseline_ad),

    "supported_advertisers":
        us_baseline_ad["assignment_date"].notna().sum(),

    "expired_advertisers":
        expired_count,

    "expiration_rate":
        expiration_rate,

    "average_wait_days":
        avg_wait,

    "p90_wait_days":
        p90_wait,

    "average_utilization":
        average_utilization,

    "total_staffing_cost":
        baseline_staffing_plan["Total_Cost"].sum()
}

In [143]:
staffing_decisions = pd.DataFrame({
    "Country": ["USA"] * 12,

    "Month": pd.period_range(
        "2024-01",
        "2024-12",
        freq="M"
    ),

    "Hires": [
        5, 3, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0
    ],

    "Terminations": [
        0, 0, 2, 0,
        0, 0, 0, 0,
        0, 0, 0, 0
    ]
})
staffing_decisions

,Country,Month,Hires,Terminations
0,USA,2024-01,5,0
1,USA,2024-02,3,0
2,USA,2024-03,0,2
3,USA,2024-04,0,0
4,USA,2024-05,0,0
5,USA,2024-06,0,0
6,USA,2024-07,0,0
7,USA,2024-08,0,0
8,USA,2024-09,0,0
9,USA,2024-10,0,0


In [178]:
staffing_plan = build_staffing_schedule(
    initial_agents=50,
    decisions=staffing_decisions
)
staffing_plan.head()

,Country,Month,Agents,Agents_Ramping,Hires,Terminations_Notices,Annual_Agent_Salary_USD,Monthly_Salary,Paid_Agents,Salary_Cost,Termination_Cost,Total_Cost
0,USA,2024-01,50,5,5,0,77721,6476.75,55,356221.25,0.0,356221.25
1,USA,2024-02,55,3,3,0,77721,6476.75,58,375651.50,0.0,375651.50
2,USA,2024-03,58,0,0,2,77721,6476.75,58,375651.50,62176.8,437828.30
3,USA,2024-04,56,0,0,0,77721,6476.75,56,362698.00,0.0,362698.00
4,USA,2024-05,56,0,0,0,77721,6476.75,56,362698.00,0.0,362698.00


In [95]:
all_daily = []
all_advertisers = []

for _, row in df_exist_staff.iterrows():

    country = row["Country"]
    agents = int(row["Existing_Agent_Count"])

    advertiser_result, daily_result = simulate_country(eligible_ad, country=country, agent_count=agents)

    all_advertisers.append(advertiser_result)

    all_daily.append(daily_result)

simulation_daily = pd.concat(all_daily, ignore_index=True)
simulation_advertisers = pd.concat(all_advertisers,ignore_index=True) 

In [99]:
#simulation_daily.tail()
#simulation_advertisers.tail()

,Advertiser_ID,Country,Sign_Up_Date,Projected_Annual_Budget,Advertiser_Eligibility_Threshold_USD,Eligible,Month,status,assignment_date,graduation_date,expiration_date
211932,A498268,UAE,2025-12-29,1590,1000,True,2025-12,NOT_ARRIVED,NaT,NaT,NaT
211933,A498644,UAE,2025-12-30,3522,1000,True,2025-12,NOT_ARRIVED,NaT,NaT,NaT
211934,A499639,UAE,2025-12-31,2982,1000,True,2025-12,NOT_ARRIVED,NaT,NaT,NaT
211935,A499640,UAE,2025-12-31,1338,1000,True,2025-12,NOT_ARRIVED,NaT,NaT,NaT
211936,A499644,UAE,2025-12-31,4318,1000,True,2025-12,NOT_ARRIVED,NaT,NaT,NaT


12. Implement hiring
13. Implement 1-month ramp-up
14. Implement firing notice
15. Implement 1-month firing delay
16. Implement salary costs
17. Implement 40% termination cost
18. 18. Build PuLP model
19. Generate monthly staffing recommendation
20. Feed recommendation into daily simulator
21. Calculate wait/utilization/expiration
22. Adjust optimization if service target fails
23. 23. Base scenario
24. -20% demand
25. +20% demand
26. +40% stress scenario
27. Compare financial/service metrics
28. 28. Define Pydantic Scenario model
29. Add LLM structured output
30. Build scenario interpretation prompt
31. Build explanation prompt
32. Build LangGraph workflow
33. Add follow-up Q&A
34. 34. Build Streamlit UI
35. Add staffing graphs
36. Add cost graphs
37. Add queue/wait graphs
38. Add sensitivity comparison
39. Add AI chat
40. Add downloadable staffing plan

In [84]:
fake_expiration = pd.DataFrame({
    "Advertiser_ID": [
        f"A{i:03d}"
        for i in range(1, 31)
    ],
    "Country": ["US"] * 30,
    "Sign_Up_Date": ["2026-01-01"] * 30,
    "Projected_Annual_Budget": [100000] * 30,
    "Eligible": [True] * 30
})

fake_expiration["Sign_Up_Date"] = pd.to_datetime(
    fake_expiration["Sign_Up_Date"]
)
print(fake_expiration)

   Advertiser_ID Country Sign_Up_Date  Projected_Annual_Budget  Eligible
0           A001      US   2026-01-01                   100000      True
1           A002      US   2026-01-01                   100000      True
2           A003      US   2026-01-01                   100000      True
3           A004      US   2026-01-01                   100000      True
4           A005      US   2026-01-01                   100000      True
5           A006      US   2026-01-01                   100000      True
6           A007      US   2026-01-01                   100000      True
7           A008      US   2026-01-01                   100000      True
8           A009      US   2026-01-01                   100000      True
9           A010      US   2026-01-01                   100000      True
10          A011      US   2026-01-01                   100000      True
11          A012      US   2026-01-01                   100000      True
12          A013      US   2026-01-01              

In [75]:
import pandas as pd	
fake_advertisers = pd.DataFrame({
    "Advertiser_ID": [
        "A001", "A002", "A003", "A004", "A005",
        "A006", "A007", "A008", "A009", "A010",
        "A011", "A012", "A013", "A014", "A015"
    ],
    "Country": ["US"] * 15,
    "Sign_Up_Date": (
        ["2026-01-01"] * 10 +
        ["2026-01-02"] * 5
    ),
    "Projected_Annual_Budget": [100000] * 15,
    "Eligible": [True] * 15
})

fake_advertisers["Sign_Up_Date"] = pd.to_datetime(
    fake_advertisers["Sign_Up_Date"]
)

print(fake_advertisers)

   Advertiser_ID Country Sign_Up_Date  Projected_Annual_Budget  Eligible
0           A001      US   2026-01-01                   100000      True
1           A002      US   2026-01-01                   100000      True
2           A003      US   2026-01-01                   100000      True
3           A004      US   2026-01-01                   100000      True
4           A005      US   2026-01-01                   100000      True
5           A006      US   2026-01-01                   100000      True
6           A007      US   2026-01-01                   100000      True
7           A008      US   2026-01-01                   100000      True
8           A009      US   2026-01-01                   100000      True
9           A010      US   2026-01-01                   100000      True
10          A011      US   2026-01-02                   100000      True
11          A012      US   2026-01-02                   100000      True
12          A013      US   2026-01-02              

In [85]:
result_advertisers, result_daily = simulate_country(
    advertisers=fake_expiration,
    country="US",
    agent_count=1,
    start_date="2026-01-01",
    end_date="2026-03-31"
)

In [86]:
 result_daily[result_daily["date"] == "2026-01-01"]

,date,country,agents,capacity,new_eligible,assigned_today,graduated_today,expired_today,supported,waiting,utilization
0,2026-01-01,US,1,10,30,10,0,0,10,20,1.0


In [87]:
 result_daily[result_daily["date"] == "2026-01-02"]

,date,country,agents,capacity,new_eligible,assigned_today,graduated_today,expired_today,supported,waiting,utilization
1,2026-01-02,US,1,10,0,0,0,0,10,20,1.0


In [88]:
print(
    result_advertisers[
        ["Advertiser_ID",
         "Sign_Up_Date",
         "assignment_date",
         "graduation_date",
         "status"]
    ]
) 

   Advertiser_ID Sign_Up_Date assignment_date graduation_date     status
0           A001   2026-01-01      2026-01-01      2026-03-02  GRADUATED
1           A002   2026-01-01      2026-01-01      2026-03-02  GRADUATED
2           A003   2026-01-01      2026-01-01      2026-03-02  GRADUATED
3           A004   2026-01-01      2026-01-01      2026-03-02  GRADUATED
4           A005   2026-01-01      2026-01-01      2026-03-02  GRADUATED
5           A006   2026-01-01      2026-01-01      2026-03-02  GRADUATED
6           A007   2026-01-01      2026-01-01      2026-03-02  GRADUATED
7           A008   2026-01-01      2026-01-01      2026-03-02  GRADUATED
8           A009   2026-01-01      2026-01-01      2026-03-02  GRADUATED
9           A010   2026-01-01      2026-01-01      2026-03-02  GRADUATED
10          A011   2026-01-01      2026-03-02      2026-05-01   ASSIGNED
11          A012   2026-01-01      2026-03-02      2026-05-01   ASSIGNED
12          A013   2026-01-01      2026-03-02      

In [69]:
result_advertisers["wait_days"] = (
    result_advertisers["assignment_date"]
    - result_advertisers["Sign_Up_Date"]
).dt.days
print(
    result_advertisers[
        [
            "Advertiser_ID",
            "Sign_Up_Date",
            "assignment_date",
            "wait_days"
        ]
    ]
)

   Advertiser_ID Sign_Up_Date assignment_date  wait_days
0           A001   2026-01-01      2026-01-01          0
1           A002   2026-01-01      2026-01-01          0
2           A003   2026-01-01      2026-01-01          0
3           A004   2026-01-01      2026-01-01          0
4           A005   2026-01-01      2026-01-01          0
5           A006   2026-01-01      2026-01-01          0
6           A007   2026-01-01      2026-01-01          0
7           A008   2026-01-01      2026-01-01          0
8           A009   2026-01-01      2026-01-01          0
9           A010   2026-01-01      2026-01-01          0
10          A011   2026-01-02      2026-03-02         59
11          A012   2026-01-02      2026-03-02         59
12          A013   2026-01-02      2026-03-02         59
13          A014   2026-01-02      2026-03-02         59
14          A015   2026-01-02      2026-03-02         59
